In [0]:
import requests, io
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from collections import Counter
from datetime import date, timedelta, datetime, timezone

END = date.today() - timedelta(days=4)
START = END - timedelta(days=10)
BASE = "https://openaq-data-archive.s3.amazonaws.com/records/csv.gz"
TABLE = "workspace.delhi_air.bronze_openaq_archive"

location_ids = [r.location_id for r in spark.sql(
    "SELECT DISTINCT location_id FROM workspace.delhi_air.silver_pm25_hourly").collect()]

done = {r.source_file for r in spark.table(TABLE).select("source_file").distinct().collect()}

jobs = []
d = START
while d <= END:
    for loc in location_ids:
        name = f"location-{loc}-{d:%Y%m%d}.csv.gz"
        if name not in done:
            jobs.append((loc, d, name))
    d += timedelta(days=1)
print("window:", START, "to", END, "files to fetch:", len(jobs))

def fetch(job):
    loc, d, name = job
    url = f"{BASE}/locationid={loc}/year={d:%Y}/month={d:%m}/{name}"
    try:
        r = requests.get(url, timeout=30)
    except Exception:
        return name, None, "error"
    if r.status_code != 200:
        return name, None, r.status_code
    df = pd.read_csv(io.BytesIO(r.content), compression="gzip")
    df = df[df["parameter"] == "pm25"].copy()
    df["source_file"] = name
    return name, df, 200

with ThreadPoolExecutor(max_workers=16) as pool:
    results = list(pool.map(fetch, jobs))

print("statuses:", Counter(status for _, _, status in results))
frames = [df for _, df, status in results if status == 200 and len(df) > 0]

if frames:
    all_df = pd.concat(frames, ignore_index=True)
    all_df = all_df.rename(columns={"sensors_id": "sensor_id", "datetime": "datetime_text"})
    all_df = all_df[["location_id", "sensor_id", "datetime_text", "parameter", "units", "value", "source_file"]]
    all_df["ingested_at"] = datetime.now(timezone.utc).replace(tzinfo=None)
    spark.createDataFrame(all_df).write.mode("append").saveAsTable(TABLE)
    print("saved rows:", len(all_df))
else:
    print("nothing new to save")